In [0]:
from pyspark.sql import SparkSession, DataFrame, functions as F
from pyspark.sql.window import Window
import uuid
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from delta.tables import DeltaTable



def append_with_incrementing_id(
    new_df: DataFrame,
    table_name: str,
    id_column: str = "id",
    order_by_column: str = None,
    database: str = "default"
):
    """
    Appends new_df to a Delta table with string auto-increment IDs like '1', '2', '3', ...

    Parameters:
    - new_df: The new data to insert.
    - table_name: Target Delta table name (must exist).
    - id_column: Name of the ID column (default: 'id').
    - order_by_column: Optional: Column to use for ordering (for deterministic IDs).
    - database: Databricks database containing the table.
    """
    spark = SparkSession.getActiveSession()
    full_table_name = f"{database}.{table_name}"

    if not spark._jsparkSession.catalog().tableExists(full_table_name):
        raise Exception(f"Table {full_table_name} does not exist. Please create it first.")

    existing_df = spark.table(full_table_name)

    if id_column in existing_df.columns:
        numeric_part_expr = F.regexp_extract(F.col(id_column), r"(\d+)$", 1).cast("long")
        max_id_row = existing_df.select(F.max(numeric_part_expr)).collect()[0][0]
        max_id = max_id_row if max_id_row is not None else 0
    else:
        max_id = 0

    if order_by_column and order_by_column in new_df.columns:
        windowSpec = Window.orderBy(F.col(order_by_column))
    else:
        windowSpec = Window.orderBy(F.lit(1))

    new_df_with_number = new_df.withColumn(
        "__rownum", F.row_number().over(windowSpec) + max_id
    )

    new_df_with_id = new_df_with_number.withColumn(
        id_column, F.col("__rownum")
    ).drop("__rownum")

    cols = new_df_with_id.columns
    ordered_cols = [id_column] + [c for c in cols if c != id_column]
    new_df_with_id = new_df_with_id.select(ordered_cols)

    # new_df_with_id.write.mode("append").saveAsTable(full_table_name)

    new_df_with_id.printSchema()
    
    new_df_with_id.write.format("delta").mode("append").option("mergeSchema", "false").insertInto(full_table_name)  

    print(f"✅ Appended {new_df_with_id.count()} rows to {full_table_name} with IDs like '1', '2', '3', ...")


def start_run_cycle(
    packagename: str,
):
    """
    Inserts a new row into the run cycle table to mark the start of a cycle.

    Parameters:
    - table_name: name of the target Delta table
    - description: description of this run cycle (string)
    - packageid: package identifier (string)
    - packagename: package name (string)
    - database: optional database name (default: 'default')
    """
    spark = SparkSession.getActiveSession()
    df_existing = spark.sql("SELECT max(cast(runcycleid as int)) as runcycleid FROM dimruncycle")
    runcycleid = df_existing.first().runcycleid + 1
    full_table = f"default.dimruncycle"

    description = "package: " + packagename + " started"

    # Generate a UUID object
    uuid_obj = uuid.uuid4()

    # Convert the UUID object to a string and make it uppercase
    packageid = str(uuid_obj).upper()

    # Build single-row DataFrame
    data = spark.createDataFrame(
        [
            (
                runcycleid,
                None,  # runcyclestartat (will be filled below)
                description,
                packageid,
                packagename,
                None,  # runcycleendat
                "NULL"  # success
            )
        ],
        schema = StructType([
                StructField("runcycleid", IntegerType(), False),         # int
                StructField("runcyclestartat", TimestampType(), True),   # timestamp
                StructField("description", StringType(), True),          # string
                StructField("packageid", StringType(), True),            # string
                StructField("packagename", StringType(), True),          # string
                StructField("runcycleendat", StringType(), True),        # string (you may want to make this a TimestampType too)
                StructField("success", StringType(), True),              # string
        ])
    )

    # Set current timestamp as runcyclestartat
    df_with_timestamp = data.withColumn("runcyclestartat", current_timestamp())

    # Append to table
    df_with_timestamp.write.mode("append").saveAsTable(full_table)

    print(f"✅ Run cycle '{runcycleid}' inserted into {full_table}.")
    return runcycleid

def end_run_cycle(
    runcycleid: str,
    success: str,
    packagename: str,
    error: str = None,
):
    """
    Updates the run cycle row to mark the end of the run.

    Parameters:
    - table_name: name of the target Delta table
    - runcycleid: ID of the run cycle to update
    - success: True or False indicating run success
    - database: optional database name (default: 'default')
    """
    spark = SparkSession.getActiveSession()
    full_table = f"default.dimruncycle"

    delta_table = DeltaTable.forName(spark, full_table)

    if success == 't':
        description = "package: " + packagename + " complete"
    else:        
        description = "package: " + packagename + " error " + error

    

    # Perform update
    delta_table.update(
        condition=f"runcycleid = '{runcycleid}'",
        set={
            "description": lit(str(description)),
            "runcycleendat": current_timestamp().cast("string"),
            "success": lit(str(success).lower()),
        }
    )

    print(f"✅ Run cycle '{runcycleid}' marked as ended with success={success}.")

In [0]:
# src/utils/job_tracker.py

import uuid
from datetime import datetime, timedelta
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, max
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

DEFAULT_START_DATE = datetime(2025, 7, 11).date()

def _ensure_job_tracker_table_exists(spark: SparkSession, job_tracker_table_path: str):
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS hive_metastore.default.job_tracker (
            job_name STRING NOT NULL,
            run_id STRING NOT NULL,
            start_time TIMESTAMP NOT NULL,
            end_time TIMESTAMP,
            status STRING NOT NULL,
            message STRING
        ) USING DELTA
        LOCATION '{job_tracker_table_path}'
    """
    try:
        spark.sql(create_table_sql)
        print(f"Ensured job tracker table exists at: {job_tracker_table_path}")
    except Exception as e:
        print(f"Error ensuring job tracker table exists: {e}")
        raise # Re-raise to prevent job from proceeding without tracker


def get_last_successful_run_time(spark: SparkSession, job_tracker_table_path: str, job_name: str) -> datetime | None:
    try:
        _ensure_job_tracker_table_exists(spark, job_tracker_table_path)
        tracker_df = spark.read.format("delta").load(job_tracker_table_path)

        last_run_df = tracker_df.filter(
            (col("job_name") == job_name) & (col("status") == "SUCCEEDED")
        ).orderBy(col("start_time").desc())

        if last_run_df.count() > 0:
            last_successful_time = last_run_df.first()["start_time"]
            print(f"Found last successful run for '{job_name}' at: {last_successful_time}")
            return last_successful_time
        else:
            print(f"No previous successful run found for '{job_name}'.")
            return None
    except Exception as e:
        print(f"Error reading job tracker for last successful run for '{job_name}': {e}")
        return None


def record_job_status(spark: SparkSession, job_tracker_table_path: str, job_name: str, run_id: str, status: str,
                      start_time: datetime, end_time: datetime = None,
                      message: str = None):
    _ensure_job_tracker_table_exists(spark, job_tracker_table_path) # Ensure table before writing

    schema = StructType([
        StructField("job_name", StringType(), False),
        StructField("run_id", StringType(), False),
        StructField("start_time", TimestampType(), False),
        StructField("end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("message", StringType(), True)
    ])

    data = [(job_name, run_id, start_time, end_time, status, message)]

    new_status_df = spark.createDataFrame(data, schema=schema)
    new_status_df.createOrReplaceTempView("new_status_df_temp_view") # Create a temp view for MERGE

    try:
        merge_sql = f"""
            MERGE INTO delta.`{job_tracker_table_path}` AS target
            USING new_status_df_temp_view AS source
            ON target.job_name = source.job_name AND target.run_id = source.run_id
            WHEN MATCHED THEN
                UPDATE SET
                    end_time = source.end_time,
                    status = source.status,
                    message = source.message
            WHEN NOT MATCHED THEN
                INSERT (job_name, run_id, start_time, end_time, status, message)
                VALUES (source.job_name, source.run_id, source.start_time, source.end_time, source.status, source.message)
        """
        spark.sql(merge_sql)
        print(f"Job status for '{job_name}' (run_id: {run_id}) recorded as: {status}")
    except Exception as e:
        print(f"ERROR: Failed to record job status for '{job_name}' (run_id: {run_id}): {e}")
        raise # Re-raise to ensure the job failure is propagated

def get_current_run_id(spark: SparkSession) -> str:
    """
    Get the Databricks run ID from Spark conf, otherwise generates a UUID.
    """
    try:
        return spark.conf.get("spark.databricks.driver.runId")
    except Exception:
        run_id = str(uuid.uuid4())
        print(f"Warning: spark.databricks.driver.runId not found. Using generated UUID: {run_id}")
        return run_id

def generate_date_range_json(last_successful_run_date: datetime | None, current_job_date: datetime) -> list[str]:
    """
    Generates a JSON array of dates (YYYY-MM-DD) between the last successful run date and the current job date.
    """
    date_list = []
    
    # Ensure it's date only, not time
    end_date = current_job_date.date()

    if last_successful_run_date:
        # start from a day before the last successful run date
        start_date = last_successful_run_date.date()
        # Ensure start_date is not after end_date
        if start_date > end_date:
            start_date = end_date
    else:
        # If no last successful run, start from DEFAULT_START_DATE.
        start_date = DEFAULT_START_DATE

    current_date = start_date
    while current_date <= end_date:
        date_list.append(current_date.strftime("%Y-%m-%d"))
        current_date += timedelta(days=1)

    return date_list

In [0]:
%sql

-- Drop the function if it already exists to allow recreation during development
DROP FUNCTION IF EXISTS udf_get_workdays_bc;

-- Create the SQL UDF
CREATE FUNCTION udf_get_workdays_bc(
    startDate DATE,
    endDate DATE,
    bcHolidays ARRAY<DATE> DEFAULT ARRAY() -- Optional parameter, defaults to an empty array
)
RETURNS INT
RETURN (
    CASE
        WHEN startDate IS NULL OR endDate IS NULL THEN 0
        WHEN endDate < startDate THEN 0
        ELSE
            REDUCE(
                -- 1. Generate a sequence of all dates between startDate and endDate
                SEQUENCE(startDate, endDate, INTERVAL 1 DAY),
                -- 2. Initialize the accumulator (workday count) to 0
                0,
                -- 3. Lambda function to process each date in the sequence
                (acc, current_date) -> acc + CASE
                                            -- Check if it's a weekend (Sunday=1, Saturday=7 in DAYOFWEEK)
                                            WHEN DAYOFWEEK(current_date) IN (1, 7) THEN 0
                                            -- Check if the current_date is in the provided bcHolidays array
                                            WHEN ARRAY_CONTAINS(bcHolidays, current_date) THEN 0
                                            -- If not a weekend and not a holiday, it's a workday
                                            ELSE 1
                                        END
            )
    END
);


In [0]:
%restart_python
%pip install boto3
import boto3
import os
from botocore.exceptions import NoCredentialsError
import datetime
import sys
sys.path.insert(0, '/Workspace/Shared')
# import etl_helpers
from pyspark.sql.functions import lit, col

tablename = "factrequestdetails"
runcycleid = start_run_cycle(tablename)
os.makedirs("/dbfs/foi/dataload", exist_ok=True)  # make sure directory exists

try:

    df_lastrun = spark.sql(f"SELECT runcyclestartat as createddate FROM dimruncycle WHERE packagename = \"{tablename}\" AND success = 't' ORDER BY runcycleid DESC LIMIT 1")
    
    # if df_lastrun.count() > 0:
    #     lastruntime = df_lastrun.first().createddate.strftime("%Y-%m-%d %H:%M:%S")
    # else:
    #     lastruntime = "2019-01-01 00:00:00"
    # print(lastruntime)

    # in order to get daily updated data (processeddate, countontime, etc)
    # calculate from beginning to update all FOIMOD requests
    lastruntime = "2019-01-01 00:00:00"

    query = f"""
        SELECT DISTINCT
            {runcycleid} as runcycleid,
            r.foirequestid,
            CASE
                WHEN r.requesttype = 'personal' THEN 33
            ELSE
                31
            END AS requesttypeid,
            CASE
                WHEN mr.requeststatusid IS NOT NULL THEN mr.requeststatusid
            ELSE
                (SELECT requeststatusid FROM foi_mod.foirequeststatuses WHERE name = rr.status LIMIT 1)
            END AS requeststatusid,
            TRY_CAST(r.receivedmodeid AS STRING) AS receivedmodeid,
            TRY_CAST(r.deliverymodeid AS STRING) AS deliverymodeid,
            r.applicantcategoryid,
            TRY_TO_TIMESTAMP(r.receiveddate, 'yyyy-MM-dd HH:mm:ss') AS requesteddate,
            TRY_TO_TIMESTAMP(r.receiveddate, 'yyyy-MM-dd HH:mm:ss') AS receiveddate,
            TRY_TO_TIMESTAMP(mr.startdate, 'yyyy-MM-dd HH:mm:ss') AS startdate,
            CASE
                WHEN first_rawrequest.created_at IS NOT NULL THEN TRY_TO_TIMESTAMP(first_rawrequest.created_at, 'yyyy-MM-dd HH:mm:ss')
            ELSE
                TRY_TO_TIMESTAMP(mr.startdate, 'yyyy-MM-dd HH:mm:ss')
            END AS createddate,
            TRY_TO_TIMESTAMP(r.created_at, 'yyyy-MM-dd HH:mm:ss') AS modifieddate,
            CASE
                WHEN mr.assignedto IS NOT NULL THEN TRY_TO_TIMESTAMP(ministryassignedto.time_assignedto_changed_to_current_value, 'yyyy-MM-dd HH:mm:ss')
            ELSE
                TRY_TO_TIMESTAMP(rawassignedto.time_assignedto_changed_to_current_value, 'yyyy-MM-dd HH:mm:ss')
            END AS assigneddate,
            CAST(NULL AS STRING) AS receivedbyusername,
            r.createdby AS modifiedbyusername,
            CAST(NULL AS INTEGER) AS assignedbyid,
            CAST(NULL AS INTEGER) AS assignedtoid,
            CASE
                WHEN mr.foiministryrequestid IS NOT NULL THEN mr.assignedto 
            ELSE
                rr.assignedto
            END AS primaryusername,
            0 AS primarygroupid,
            CASE
                WHEN mr.assignedgroup IS NOT NULL THEN mr.assignedgroup 
            ELSE
                rr.assignedgroup
            END AS groupname,
            office.bcgovcode AS officeid,
            mr.assignedministrygroup AS ministry,
            subj.name AS subject,
            CAST(NULL AS TIMESTAMP) AS screeneddate,
            TRY_CAST(mr.duedate AS STRING) AS targetdate,
            CAST(NULL AS STRING) AS amendment,
            CAST(NULL AS TIMESTAMP) AS amendmentcreateddate,
            CAST(NULL AS STRING) AS amendmentcreatedby,
            CAST(NULL AS TIMESTAMP) AS completeddate,
            CAST(NULL AS STRING) AS completedby,
            TRY_TO_TIMESTAMP(pkg.createdat, 'yyyy-MM-dd HH:mm:ss') AS deliverydate,
            pkg.createdby AS deliveredby,
            CASE
                WHEN mr.requeststatuslabel = 'closed' AND TRY_CAST(finalclosedate.closedate AS DATE) IS NOT NULL THEN TRY_TO_TIMESTAMP(finalclosedate.closedate, 'yyyy-MM-dd HH:mm:ss')
                WHEN TRY_CAST(rr.closedate AS DATE) IS NOT NULL THEN TRY_TO_TIMESTAMP(rr.closedate, 'yyyy-MM-dd HH:mm:ss')
            ELSE
                CAST(NULL AS TIMESTAMP)
            END AS closeddate,
            CASE
                WHEN mr.requeststatuslabel = 'closed' AND TRY_CAST(finalclosedate.closedate AS DATE) IS NOT NULL THEN finalclosedate.createdby
                WHEN TRY_CAST(rr.closedate AS DATE) IS NOT NULL THEN rr.createdby
            ELSE
                NULL
            END AS closedby,
            rr.notes,
            CAST(NULL AS STRING) AS withheldstage,
            CAST(NULL AS STRING) AS otheraddress,
            CAST(NULL AS STRING) AS holdstage,
            TRY_TO_TIMESTAMP(firstonhold.created_at, 'yyyy-MM-dd HH:mm:ss') AS holddate,
            CAST(NULL AS STRING) AS appealtype,
            CAST(NULL AS STRING) AS reviewid,
            CAST(NULL AS TIMESTAMP) AS withhelddate,
            cr.name AS disposition,
            CAST(NULL AS STRING) AS execcomments,
            TRY_TO_TIMESTAMP(firstministryrequest.duedate, 'yyyy-MM-dd HH:mm:ss') AS originaltargetdate,
            CAST(NULL AS STRING) AS redactiondescription,
            CASE
                WHEN mr.requeststatuslabel IS NOT NULL THEN mr.requeststatuslabel 
            ELSE
                rr.requeststatuslabel
            END AS currentactivity,
            custom.onholddays,
            custom.onholdotherdays,
            custom.processeddays,
            CAST(NULL AS STRING) AS releaseformat,
            CAST(NULL AS STRING) AS denialauthority,
            CAST(NULL AS STRING) AS caseowner,
            CAST(NULL AS STRING) AS caseownertitle,
            CAST(NULL AS STRING) AS caseowneremail,
            CAST(NULL AS STRING) AS caseownerphone,
            TRY_TO_TIMESTAMP(firstclosedate.closedate, 'yyyy-MM-dd HH:mm:ss') AS originalcloseddate,
            TRY_TO_TIMESTAMP(mr.duedate, 'yyyy-MM-dd HH:mm:ss') AS stdduedate,
            custom.remainingdays,
            TRY_CAST(NULL AS INTEGER) AS requestage,
            TRY_TO_TIMESTAMP(currentactivitydate.created_at, 'yyyy-MM-dd HH:mm:ss') AS currentactivitydate,
            CAST(NULL AS STRING) AS crossgovtno,
            mr.description AS requestdescription,
            CASE
                WHEN mr.requeststatusid IS NOT NULL THEN (SELECT name FROM foi_mod.foirequeststatuses WHERE requeststatusid = mr.requeststatusid LIMIT 1) 
            ELSE
                rr.status
            END AS requeststatus,
            CASE
                WHEN mr.requeststatusid IS NOT NULL THEN (SELECT description FROM foi_mod.foirequeststatuses WHERE requeststatusid = mr.requeststatusid LIMIT 1) 
            ELSE
                rr.status
            END AS status,
            custom.countontime,
            custom.countoverdue,
            custom.daysoverdue,
            oistatus.name AS publication,
            oie.name AS publicationreason,
            latestoipc.oipcno,
            latestoipc.isjudicialreview AS judicialreview,
            oipctypes.name AS reviewtype,
            reason.name AS reason,
            latestoipc.investigator AS portfolioofficer,
            custom.passduedays,
            mr.description AS description,
            CAST(NULL AS STRING) AS priority,
            TRY_CAST(applicant.foirequestapplicantid AS INTEGER) AS requesterid,
            TRY_CAST(onbehalf.foirequestapplicantid AS STRING) AS onbehalfofrequesterid,
            CAST(NULL AS STRING) AS referencenumber,
            CAST(NULL AS STRING) AS refvisualrequestfilenumber,
            TRY_CAST(ext.extension AS STRING) AS extension,
            CAST(NULL AS INTEGER) shipaddressid,
            CAST(NULL AS INTEGER) AS billaddressid,
            TRY_TO_TIMESTAMP(r.receiveddate, 'yyyy-MM-dd HH:mm:ss') AS originalreceiveddate,
            CAST(NULL AS STRING) AS coordinatednrresponsereqd,
            CAST(NULL AS STRING) AS applicantfilereference,
            custom.secondaryusers,
            custom.noofdocdelivered,
            CASE
                WHEN r.isactive = TRUE THEN 'Y'
            ELSE
                'N'
            END AS activeflag,
            CAST(NULL AS STRING) AS customfieldstatus,
            CASE
                WHEN mr.axisrequestid IS NOT NULL THEN RTRIM(mr.axisrequestid)
            ELSE
                RTRIM(rr.axisrequestid)
            END AS visualrequestfilenumber,
            'FOIMOD' AS sourceoftruth,
            CASE
                WHEN mr.assignedto IS NOT NULL THEN ministryassignedto.created_by_when_changed_assignedto
            ELSE
                rawassignedto.created_by_when_changed_assignedto
            END AS assignedby,
            CASE
                WHEN mr.assignedto IS NOT NULL THEN mr.assignedto 
            ELSE
                rr.assignedto
            END AS assignedto,
            CASE 
                WHEN mr.isconsultflag IS NULL OR LOWER(mr.isconsultflag) IN ('f', 'false', 'null') THEN 'N'
                WHEN LOWER(mr.isconsultflag) IN ('t', 'true') THEN 'Y'
                ELSE 'N'
            END AS isconsultflag,
            CASE 
                WHEN mr.isphasedrelease IS NULL OR LOWER(mr.isphasedrelease) IN ('f', 'false', 'null') THEN 'N'
                WHEN LOWER(mr.isphasedrelease) IN ('t', 'true') THEN 'Y'
                ELSE 'N'
            END AS isphasedrelease,
            CASE 
                WHEN mr.isoipcreview IS NULL OR LOWER(mr.isoipcreview) IN ('f', 'false', 'null') THEN 'N'
                WHEN LOWER(mr.isoipcreview) IN ('t', 'true') THEN 'Y'
                ELSE 'N'
            END AS isoipcreview,
            CASE
                WHEN iaorestricted.isrestricted IN ('t', 'true') THEN 'Y'
                ELSE 'N'
            END AS isiaorestricted,
            CASE
                WHEN ministryrestricted.isrestricted IN ('t', 'true') THEN 'Y'
                ELSE 'N'
            END AS isministryrestricted,
            mr.foiministryrequestid
        FROM 
        (
          SELECT *
          FROM foi_mod.foiministryrequests
          WHERE isactive = true
          QUALIFY ROW_NUMBER() OVER (
              PARTITION BY foiministryrequestid ORDER BY version DESC
          ) = 1
        ) AS mr
        INNER JOIN foi_mod.foirequests r ON r.foirequestid = mr.foirequest_id AND r.version = mr.foirequestversion_id
        LEFT JOIN (
            SELECT
                *,
                ROW_NUMBER() OVER(PARTITION BY requestid ORDER BY version DESC) as rn
            FROM
                foi_mod.foirawrequests
        ) AS rr ON r.foirawrequestid = rr.requestid AND rr.rn = 1
        LEFT JOIN (
            SELECT
                requestid,
                TRY_CAST(created_at AS DATE) AS created_at
            FROM foi_mod.foirawrequests
            WHERE version = 1
        ) first_rawrequest ON r.foirawrequestid = first_rawrequest.requestid
        LEFT JOIN (
            WITH LatestAssignedTo AS (
                SELECT
                    foiministryrequestid,
                    foirequest_id,
                    assignedto AS current_assignedto_value
                FROM
                    foi_mod.foiministryrequests
                    QUALIFY ROW_NUMBER() OVER (
                        PARTITION BY foiministryrequestid ORDER BY version DESC
                    ) = 1
            ),
            FirstOccurrenceOfCurrentAssignment AS (
                SELECT
                    t.foiministryrequestid,
                    t.foirequest_id,
                    TRY_CAST(t.created_at AS DATE) AS time_assignedto_changed_to_current_value,
                    t.createdby AS created_by_when_changed_assignedto
                FROM
                    foi_mod.foiministryrequests AS t
                INNER JOIN
                    LatestAssignedTo AS lat ON t.foiministryrequestid = lat.foiministryrequestid AND t.assignedto = lat.current_assignedto_value
                    QUALIFY ROW_NUMBER() OVER (
                        PARTITION BY t.foiministryrequestid ORDER BY t.version ASC
                    ) = 1
            )
            SELECT
                lat.foiministryrequestid,
                lat.foirequest_id,
                lat.current_assignedto_value AS latest_assignedto,
                fo.time_assignedto_changed_to_current_value,
                fo.created_by_when_changed_assignedto
            FROM LatestAssignedTo AS lat
            INNER JOIN FirstOccurrenceOfCurrentAssignment AS fo ON lat.foiministryrequestid = fo.foiministryrequestid
        ) ministryassignedto on ministryassignedto.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            WITH LatestAssignedTo AS (
                SELECT
                    requestid,
                    assignedto AS current_assignedto_value
                FROM
                    foi_mod.foirawrequests
                    QUALIFY ROW_NUMBER() OVER (
                        PARTITION BY requestid ORDER BY version DESC
                    ) = 1
            ),
            FirstOccurrenceOfCurrentAssignment AS (
                SELECT
                    t.requestid,
                    TRY_CAST(t.created_at AS DATE) AS time_assignedto_changed_to_current_value,
                    t.createdby AS created_by_when_changed_assignedto
                FROM
                    foi_mod.foirawrequests AS t
                INNER JOIN
                    LatestAssignedTo AS lat ON t.requestid = lat.requestid AND t.assignedto = lat.current_assignedto_value
                    QUALIFY ROW_NUMBER() OVER (
                        PARTITION BY t.requestid ORDER BY t.version ASC
                    ) = 1
            )
            SELECT
                lat.requestid,
                lat.current_assignedto_value AS latest_assignedto,
                fo.time_assignedto_changed_to_current_value,
                fo.created_by_when_changed_assignedto
            FROM LatestAssignedTo AS lat
            INNER JOIN FirstOccurrenceOfCurrentAssignment AS fo
            ON lat.requestid = fo.requestid
        ) rawassignedto on rawassignedto.requestid = rr.requestid
        LEFT JOIN foi_mod.foiministryrequestsubjectcodes msubj on msubj.foiministryrequestid = mr.foiministryrequestid AND msubj.foiministryrequestversion = mr.version
        LEFT JOIN foi_mod.subjectcodes subj on subj.subjectcodeid = msubj.subjectcodeid
        LEFT JOIN (
            SELECT
                ministryrequestid,
                TRY_CAST(createdat AS DATE) AS createdat,
                createdby
            FROM docreviewer.pdfstitchpackage
            QUALIFY ROW_NUMBER() OVER (PARTITION BY ministryrequestid ORDER BY pdfstitchpackageid DESC) = 1
        ) pkg on pkg.ministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                foiministryrequestid,
                TRY_CAST(created_at AS DATE) AS created_at
            FROM foi_mod.foiministryrequests
            WHERE requeststatuslabel = 'onhold'
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequestid ORDER BY version ASC) = 1
        ) firstonhold on firstonhold.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                foiministryrequest_id,
                foiministryrequestversion_id,
                oipcno,
                isjudicialreview,
                investigator,
                reviewtypeid,
                reasonid,
                created_at
            FROM foi_mod.foirequestoipc
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequest_id ORDER BY foiministryrequestversion_id DESC) = 1
        ) latestoipc on latestoipc.foiministryrequest_id = mr.foiministryrequestid AND latestoipc.foiministryrequestversion_id = mr.version
        LEFT JOIN foi_mod.oipcreviewtypes oipctypes on oipctypes.reviewtypeid = latestoipc.reviewtypeid
        LEFT JOIN foi_mod.oipcreasons reason on reason.reasonid = latestoipc.reasonid
        INNER JOIN foi_mod.foirequestapplicantmappings applicant on applicant.foirequest_id = r.foirequestid AND applicant.foirequestversion_id = r.version AND applicant.requestortypeid = 1
        LEFT JOIN foi_mod.foirequestapplicantmappings onbehalf on onbehalf.foirequest_id = r.foirequestid AND onbehalf.foirequestversion_id = r.version AND onbehalf.requestortypeid = 2
        LEFT JOIN (
            SELECT
                foiministryrequestid,
                TRY_CAST(created_at AS DATE) AS created_at
            FROM foi_mod.foiministryrequests
            WHERE duedate IS NULL
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequestid ORDER BY version ASC) = 1
        ) firstduedate on firstduedate.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                foiministryrequestid,
                TRY_CAST(duedate AS DATE) AS duedate
            FROM foi_mod.foiministryrequests
            WHERE version = 1
        ) firstministryrequest on firstministryrequest.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                foiministryrequestid,
                TRY_CAST(closedate AS DATE) AS closedate
            FROM foi_mod.foiministryrequests
            WHERE closedate IS NOT NULL AND closedate != 'NULL'
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequestid ORDER BY version ASC) = 1
        ) firstclosedate on firstclosedate.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                foiministryrequestid,
                createdby,
                TRY_CAST(closedate AS DATE) AS closedate
            FROM foi_mod.foiministryrequests
            WHERE closedate IS NOT NULL AND closedate != 'NULL'
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequestid ORDER BY version DESC) = 1
        ) finalclosedate on finalclosedate.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            WITH LatestStatus AS (
                SELECT
                    foiministryrequestid,
                    requeststatuslabel AS status
                FROM
                    foi_mod.foiministryrequests
                    QUALIFY ROW_NUMBER() OVER (
                        PARTITION BY foiministryrequestid ORDER BY version DESC
                    ) = 1
            )
            SELECT
                fmr.foiministryrequestid,
                fmr.created_at
            FROM foi_mod.foiministryrequests fmr
            INNER JOIN LatestStatus ON fmr.foiministryrequestid = LatestStatus.foiministryrequestid AND fmr.requeststatuslabel = LatestStatus.status
            QUALIFY ROW_NUMBER() OVER (PARTITION BY fmr.foiministryrequestid ORDER BY fmr.version ASC) = 1
        ) currentactivitydate on currentactivitydate.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                foiministryrequest_id,
                foiministryrequestversion_id,
                SUM(COALESCE(TRY_CAST(approvednoofdays AS INT), 0)) AS extension
            FROM foi_mod.foirequestextensions
            WHERE isactive = 't'
            GROUP BY foiministryrequest_id, foiministryrequestversion_id
        ) ext ON ext.foiministryrequest_id = mr.foiministryrequestid AND ext.foiministryrequestversion_id = mr.version
        LEFT JOIN foi_mod.programareas office on office.programareaid = mr.programareaid
        LEFT JOIN (
            SELECT
                foiministryrequest_id,
                oiexemption_id,
                oipublicationstatus_id,
                created_at
            FROM
                foi_mod.foiopeninformationrequests
                QUALIFY ROW_NUMBER() OVER (
                    PARTITION BY foiministryrequest_id ORDER BY version DESC
                ) = 1
        ) oi ON oi.foiministryrequest_id = mr.foiministryrequestid
        LEFT JOIN foi_mod.openinfopublicationstatuses oistatus ON oistatus.oipublicationstatusid = oi.oipublicationstatus_id
        LEFT JOIN foi_mod.openinformationexemptions oie ON oie.oiexemptionid = oi.oiexemption_id
        LEFT JOIN foi_mod.closereasons cr ON try_cast(cr.closereasonid as bigint) = try_cast(mr.closereasonid as bigint)
        LEFT JOIN default.factrequestcustomcalcfields custom ON custom.foiministryrequestid = mr.foiministryrequestid and custom.sourceoftruth = 'FOIMOD'
        LEFT JOIN (
            SELECT
                _mr.foiministryrequestid,
                _rmr.isrestricted
            FROM foi_mod.foiministryrequests _mr
            INNER JOIN foi_mod.foirestrictedministryrequests _rmr on _mr.foiministryrequestid = _rmr.ministryrequestid
            WHERE _mr.isactive in ('t', 'true') and _rmr.isactive in ('t', 'true') and _rmr.isrestricted in ('t', 'true') and _rmr.type = 'iao'
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY _mr.foiministryrequestid ORDER BY _mr.version DESC
            ) = 1
        ) iaorestricted ON iaorestricted.foiministryrequestid = mr.foiministryrequestid
        LEFT JOIN (
            SELECT
                _mr.foiministryrequestid,
                _rmr.isrestricted
            FROM foi_mod.foiministryrequests _mr
            INNER JOIN foi_mod.foirestrictedministryrequests _rmr on _mr.foiministryrequestid = _rmr.ministryrequestid
            WHERE _mr.isactive in ('t', 'true') and _rmr.isactive in ('t', 'true') and _rmr.isrestricted in ('t', 'true') and _rmr.type = 'ministry'
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY _mr.foiministryrequestid ORDER BY _mr.version DESC
            ) = 1
        ) ministryrestricted ON ministryrestricted.foiministryrequestid = mr.foiministryrequestid
        WHERE r.created_at > '{lastruntime}' or oi.created_at > '{lastruntime}' or latestoipc.created_at > '{lastruntime}' or pkg.createdat > '{lastruntime}'
    """

    # print(query)

    df = spark.sql(query)
    df.show()

    # order of columns here is important!
    df_mapped = df.selectExpr(
        "runcycleid AS runcycleid",
        "foirequestid AS foirequestid",
        "requesttypeid AS requesttypeid",
        "requeststatusid AS requeststatusid",
        "receivedmodeid AS receivedmodeid",
        "deliverymodeid AS deliverymodeid",
        "applicantcategoryid AS applicantcategoryid",
        "requesteddate AS requesteddate",
        "receiveddate AS receiveddate",
        "startdate AS startdate",
        "createddate AS createddate",
        "modifieddate AS modifieddate",
        "assigneddate AS assigneddate",
        "receivedbyusername AS receivedbyusername",
        "modifiedbyusername AS modifiedbyusername",
        "assignedbyid AS assignedbyid",
        "assignedtoid AS assignedtoid",
        "primaryusername AS primaryusername",
        "primarygroupid AS primarygroupid",
        "groupname AS groupname",
        "officeid AS officeid",
        "ministry AS ministry",
        "subject AS subject",
        "screeneddate AS screeneddate",
        "targetdate AS targetdate",
        "amendment AS amendment",
        "amendmentcreateddate AS amendmentcreateddate",
        "amendmentcreatedby AS amendmentcreatedby",
        "completeddate AS completeddate",
        "completedby AS completedby",
        "deliverydate AS deliverydate",
        "deliveredby AS deliveredby",
        "closeddate AS closeddate",
        "closedby AS closedby",
        "notes AS notes",
        "withheldstage AS withheldstage",
        "otheraddress AS otheraddress",
        "holdstage AS holdstage",
        "holddate AS holddate",
        "appealtype AS appealtype",
        "reviewid AS reviewid",
        "withhelddate AS withhelddate",
        "disposition AS disposition",
        "execcomments AS execcomments",
        "originaltargetdate AS originaltargetdate",
        "redactiondescription AS redactiondescription",
        "currentactivity AS currentactivity",
        "onholddays AS onholddays",
        "onholdotherdays AS onholdotherdays",
        "processeddays AS processeddays",
        "releaseformat AS releaseformat",
        "denialauthority AS denialauthority",
        "caseowner AS caseowner",
        "caseownertitle AS caseownertitle",
        "caseowneremail AS caseowneremail",
        "caseownerphone AS caseownerphone",
        "originalcloseddate AS originalcloseddate",
        "stdduedate AS stdduedate",
        "remainingdays AS remainingdays",
        "requestage AS requestage",
        "currentactivitydate AS currentactivitydate",
        "crossgovtno AS crossgovtno",
        "requestdescription AS requestdescription",
        "requeststatus AS requeststatus",
        "status AS status",
        "countontime AS countontime",
        "countoverdue AS countoverdue",
        "daysoverdue AS daysoverdue",
        "publication AS publication",
        "publicationreason AS publicationreason",
        "oipcno AS oipcno",
        "judicialreview AS judicialreview",
        "reviewtype AS reviewtype",
        "reason AS reason",
        "portfolioofficer AS portfolioofficer",
        "passduedays AS passduedays",
        "description AS description",
        "priority AS priority",
        "requesterid AS requesterid",
        "onbehalfofrequesterid AS onbehalfofrequesterid",
        "referencenumber AS referencenumber",
        "refvisualrequestfilenumber AS refvisualrequestfilenumber",
        "extension AS extension",
        "shipaddressid AS shipaddressid",
        "billaddressid AS billaddressid",
        "originalreceiveddate AS originalreceiveddate",
        "coordinatednrresponsereqd AS coordinatednrresponsereqd",
        "applicantfilereference AS applicantfilereference",
        "secondaryusers AS secondaryusers",
        "noofdocdelivered AS noofdocdelivered",
        "activeflag AS activeflag",
        "customfieldstatus AS customfieldstatus",
        "visualrequestfilenumber AS visualrequestfilenumber",
        "sourceoftruth AS sourceoftruth",
        "assignedby AS assignedby",
        "assignedto AS assignedto",
        "isconsultflag AS isconsultflag",
        "isphasedrelease AS isphasedrelease",
        "isoipcreview AS isoipcreview",
        "isiaorestricted AS isiaorestricted",
        "isministryrestricted AS isministryrestricted",
        "foiministryrequestid AS foiministryrequestid"
    )
    df_mapped.show()

    from delta.tables import DeltaTable
    delta_table = DeltaTable.forName(spark, f"hive_metastore.default.{tablename}")
    delta_table.alias("target").merge(
        df_mapped.alias("source"),
        """
        target.sourceoftruth = source.sourceoftruth 
        AND (
            (source.foiministryrequestid IS NULL AND target.foirequestid = source.foirequestid)
            OR 
            (source.foiministryrequestid IS NOT NULL AND target.foiministryrequestid = source.foiministryrequestid)
        )
        """
    ).whenMatchedUpdate(
        condition = "target.activeflag = 'Y'",
        set = {
            "activeflag": lit("N"),
        }
    ).execute()

    print("Matched records deactivated.")

    df_mapped.write.format("delta").mode("append").saveAsTable(f"hive_metastore.default.{tablename}") 

    end_run_cycle(runcycleid, 't', tablename)
except NoCredentialsError:
    print("Credentials not available")
    end_run_cycle(runcycleid, 'f', tablename, "Credentials not available")
    raise Exception("notebook failed") from e
except Exception as e:
    if (str(e) == "no changes for today"):
        print("here")
        end_run_cycle(runcycleid, 't', tablename)
    else:
        print(f"An error occurred: {e}")    
        end_run_cycle(runcycleid, 'f', tablename, f"An error occurred: {e}")
        raise Exception("notebook failed") from e